In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.utils import Sequence
from tensorflow.keras import layers, Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers.schedules import CosineDecay
import time
import gc

print("--- STARTING KAGGLE BATCH EXECUTION: BOOSTED ALEXNET ---")

# ==========================================
# PART 1: DATA PIPELINE (MEMORY-SAFE)
# ==========================================
DATASET_PATH = "/kaggle/input/datasets/yashaswi15/aerosense-training-data/Delhi_NCR_Master_DataCube_Scaled.npy"

master_data = np.load(DATASET_PATH, mmap_mode='r')
total_days = master_data.shape[0]
train_split = int(total_days * 0.8) 
test_split = total_days - train_split 

class SpatiotemporalGenerator(Sequence):
    def __init__(self, data_cube, start_idx, end_idx, batch_size=16):
        self.data_cube = data_cube
        self.start_idx = start_idx
        self.end_idx = end_idx - 8 
        self.batch_size = batch_size
        self.indices = np.arange(self.start_idx, self.end_idx)
        
    def __len__(self):
        return int(np.ceil(len(self.indices) / self.batch_size))
    
    def __getitem__(self, idx):
        batch_indices = self.indices[idx * self.batch_size : (idx + 1) * self.batch_size]
        X_batch, Y_batch = [], []
        for i in batch_indices:
            window = self.data_cube[i : i+7]
            X_flat = np.transpose(window, (1, 2, 0, 3)).reshape((141, 231, 42))
            X_batch.append(X_flat)
            Y_batch.append(self.data_cube[i+7])
            
        X_batch = np.nan_to_num(np.array(X_batch), nan=0.0)
        Y_batch = np.nan_to_num(np.array(Y_batch), nan=0.0)
        return X_batch, Y_batch

train_gen = SpatiotemporalGenerator(master_data, 0, train_split, batch_size=16)
test_gen = SpatiotemporalGenerator(master_data, train_split, total_days, batch_size=16)

# ==========================================
# PART 2: BOOSTED ALEXNET ARCHITECTURE
# ==========================================
def build_alexnet_boosted(input_shape=(141, 231, 42)):
    inputs = layers.Input(shape=input_shape)
    
    # 1. Custom AlexNet Convolutional Base (No pre-trained weights available)
    # The aggressive 11x11 kernel instantly destroys fine spatial details
    x = layers.Conv2D(96, (11, 11), strides=(4, 4), padding='same', activation='relu')(inputs)
    x = layers.MaxPooling2D((3, 3), strides=(2, 2), padding='same')(x)
    
    x = layers.Conv2D(256, (5, 5), padding='same', activation='relu')(x)
    x = layers.MaxPooling2D((3, 3), strides=(2, 2), padding='same')(x)
    
    x = layers.Conv2D(384, (3, 3), padding='same', activation='relu')(x)
    x = layers.Conv2D(384, (3, 3), padding='same', activation='relu')(x)
    x = layers.Conv2D(256, (3, 3), padding='same', activation='relu')(x)
    x = layers.MaxPooling2D((3, 3), strides=(2, 2), padding='same')(x)
    
    # 2. Upsampling Decoder
    x = layers.Conv2DTranspose(256, (3, 3), strides=(2, 2), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2DTranspose(128, (3, 3), strides=(2, 2), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2DTranspose(64,  (3, 3), strides=(2, 2), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2DTranspose(32,  (3, 3), strides=(2, 2), padding='same', activation='relu')(x)
    x = layers.Conv2DTranspose(16,  (3, 3), strides=(2, 2), padding='same', activation='relu')(x)
    
    x = layers.Resizing(141, 231)(x)
    
    # 3. THE MAGIC SHORTCUT: Saves the model from going into negative R2
    x = layers.Concatenate()([x, inputs])
    x = layers.Conv2D(32, (3, 3), padding='same', activation='relu')(x)
    
    outputs = layers.Conv2D(6, (1, 1), activation='sigmoid')(x)
    return Model(inputs=inputs, outputs=outputs, name="Boosted_AlexNet")

alexnet_model = build_alexnet_boosted()

# ==========================================
# PART 3: ADVANCED CUSTOM TRAINING LOOP
# ==========================================
MODEL_SAVE_PATH = "/kaggle/working/Delhi_NCR_AlexNet_Best.keras"

initial_learning_rate = 0.001
decay_steps = 50 * len(train_gen)
lr_schedule = CosineDecay(initial_learning_rate, decay_steps)
optimizer = Adam(learning_rate=lr_schedule, clipnorm=1.0)

# MATCHING U-NET EXACTLY: Using MAE
def calculate_loss(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    return tf.reduce_mean(tf.abs(y_true - y_pred))

@tf.function
def train_step(x_batch, y_batch):
    with tf.GradientTape() as tape:
        predictions = alexnet_model(x_batch, training=True)
        loss = calculate_loss(y_batch, predictions)
    gradients = tape.gradient(loss, alexnet_model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, alexnet_model.trainable_variables))
    return loss

@tf.function
def val_step(x_batch, y_batch):
    predictions = alexnet_model(x_batch, training=False)
    val_loss = calculate_loss(y_batch, predictions)
    return val_loss

EPOCHS = 50
best_val_loss = float('inf')
patience = 7
patience_counter = 0

print(f"\n--- Starting 50-Epoch Backpropagation ---")

for epoch in range(EPOCHS):
    start_time = time.time()
    epoch_loss_avg = tf.keras.metrics.Mean()
    epoch_val_loss_avg = tf.keras.metrics.Mean()
    
    for step in range(len(train_gen)):
        x_batch, y_batch = train_gen[step]
        loss_val = train_step(x_batch, y_batch)
        epoch_loss_avg.update_state(loss_val)
        
    for step in range(len(test_gen)):
        x_val, y_val = test_gen[step]
        v_loss = val_step(x_val, y_val)
        epoch_val_loss_avg.update_state(v_loss)
        
    train_loss = epoch_loss_avg.result().numpy()
    val_loss = epoch_val_loss_avg.result().numpy()
    current_lr = optimizer.learning_rate(optimizer.iterations).numpy() if callable(optimizer.learning_rate) else optimizer.learning_rate.numpy()
        
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Time: {time.time() - start_time:.1f}s | LR: {current_lr:.5f} | Train Loss: {train_loss:.5f} | Val Loss: {val_loss:.5f}")
    
    if val_loss < best_val_loss:
        print(f"  -> Val Loss improved from {best_val_loss:.5f} to {val_loss:.5f}. Saving weights!")
        best_val_loss = val_loss
        alexnet_model.save(MODEL_SAVE_PATH)
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"  -> No improvement. Patience: {patience_counter}/{patience}")
        if patience_counter >= patience:
            print(f"\n[!] Early Stopping Triggered.")
            break
            
    gc.collect()
    tf.keras.backend.clear_session()

2026-04-23 14:36:21.711617: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776954981.892015      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776954981.947750      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776954982.390967      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776954982.391009      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776954982.391012      23 computation_placer.cc:177] computation placer alr

--- STARTING KAGGLE BATCH EXECUTION: BOOSTED ALEXNET ---


I0000 00:00:1776955007.047893      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0



--- Starting 50-Epoch Backpropagation ---


I0000 00:00:1776955014.046949      67 cuda_dnn.cc:529] Loaded cuDNN version 91002


Epoch 01/50 | Time: 110.2s | LR: 0.00100 | Train Loss: 0.06796 | Val Loss: 0.04979
  -> Val Loss improved from inf to 0.04979. Saving weights!
Epoch 02/50 | Time: 86.2s | LR: 0.00100 | Train Loss: 0.04633 | Val Loss: 0.04413
  -> Val Loss improved from 0.04979 to 0.04413. Saving weights!
Epoch 03/50 | Time: 87.0s | LR: 0.00099 | Train Loss: 0.04305 | Val Loss: 0.04137
  -> Val Loss improved from 0.04413 to 0.04137. Saving weights!
Epoch 04/50 | Time: 79.4s | LR: 0.00098 | Train Loss: 0.04121 | Val Loss: 0.04206
  -> No improvement. Patience: 1/7
Epoch 05/50 | Time: 78.4s | LR: 0.00098 | Train Loss: 0.03991 | Val Loss: 0.14100
  -> No improvement. Patience: 2/7
Epoch 06/50 | Time: 79.3s | LR: 0.00096 | Train Loss: 0.03893 | Val Loss: 0.03881
  -> Val Loss improved from 0.04137 to 0.03881. Saving weights!
Epoch 07/50 | Time: 79.1s | LR: 0.00095 | Train Loss: 0.03819 | Val Loss: 0.03748
  -> Val Loss improved from 0.03881 to 0.03748. Saving weights!
Epoch 08/50 | Time: 86.6s | LR: 0.00094